# Worked solutions — HTTP APIs with a local practice server

**Stage:** Advanced · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–16. No variables or files from other notebooks are required.

## What you will learn

- Make a real HTTP request with a timeout
- Decode JSON and inspect status errors
- Separate response parsing from transport


This notebook contains complete worked answers. Try the corresponding learning notebook first. These are possible solutions, not the only correct approaches. Run from the top so all examples and answer checks execute.


## 1. The request-response model

An HTTP client sends a method and URL; a server responds with a status, headers, and body. GET normally retrieves data. This tiny server listens only on your computer and serves fixed practice data, so no account or internet connection is needed. It shuts down after the demonstration.


In [1]:
import json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
from urllib.request import urlopen
from urllib.error import HTTPError

class PracticeHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/tasks":
            status = 200
            payload = {"tasks": [{"id": 1, "title": "Practice", "done": False}]}
        else:
            status = 404
            payload = {"error": "not found"}
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # Keep the educational output focused.

server = ThreadingHTTPServer(("127.0.0.1", 0), PracticeHandler)
worker = Thread(target=server.serve_forever, daemon=True)
worker.start()
base_url = f"http://127.0.0.1:{server.server_port}"
try:
    with urlopen(base_url + "/tasks", timeout=2) as response:
        print("Status:", response.status)
        payload = json.loads(response.read().decode("utf-8"))
        print(payload)
    try:
        urlopen(base_url + "/missing", timeout=2)
    except HTTPError as error:
        print("Expected missing-resource status:", error.code)
        error.close()
finally:
    server.shutdown()
    server.server_close()
    worker.join(timeout=2)
assert payload["tasks"][0]["title"] == "Practice"


Status: 200
{'tasks': [{'id': 1, 'title': 'Practice', 'done': False}]}
Expected missing-resource status: 404


**Try a change:** Change the task title in the handler and rerun the whole cell.


## 2. Build query strings

Query parameters belong in the URL after ?. urlencode handles reserved characters and spaces. Avoid manually joining unescaped user text into a URL.


In [2]:
from urllib.parse import urlencode, parse_qs
params = {"search": "python & data", "page": 2}
query = urlencode(params)
print("https://example.com/tasks?" + query)
assert parse_qs(query)["search"] == ["python & data"]


https://example.com/tasks?search=python+%26+data&page=2


**Try a change:** Try a search containing + or a non-ASCII character.


## 3. Validate the shape

Valid JSON is not necessarily the expected schema. Validate fields at a boundary before processing them. Real clients also distinguish transport failures, invalid JSON, authentication failures, rate limits, and server errors. Retry only when the operation and error make retrying appropriate.


In [3]:
def task_titles(payload):
    if not isinstance(payload, dict) or not isinstance(payload.get("tasks"), list):
        raise ValueError("expected an object containing a tasks list")
    titles = []
    for task in payload["tasks"]:
        if not isinstance(task, dict) or not isinstance(task.get("title"), str):
            raise ValueError("each task needs a string title")
        titles.append(task["title"])
    return titles

assert task_titles({"tasks": [{"title": "Read"}]}) == ["Read"]
assert task_titles({"tasks": []}) == []
print(task_titles(payload))


['Practice']


**Try a change:** Pass {'tasks': [42]} inside try/except and inspect the error.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Select completed tasks

Return titles of tasks whose done field is True. Assume tasks have valid title and boolean done fields.

<details>
<summary>Show a hint</summary>

Filter before extracting titles.

</details>


In [4]:
def completed_titles(tasks):
    return [task["title"] for task in tasks if task["done"]]


In [5]:
exercise_1_ready = True  # Set True when your implementation is ready.
if exercise_1_ready:
    assert completed_titles([{"title": "Read", "done": True}, {"title": "Code", "done": False}]) == ["Read"]
    assert completed_titles([]) == []
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: supplied checks passed. Add your own edge cases!


### Exercise 2: Flatten pages

Each response page contains an items list. Return all items in page order; treat a missing items field as an empty list.

<details>
<summary>Show a hint</summary>

extend adds the elements; append would add the list itself.

</details>


In [6]:
def collect_items(pages):
    result = []
    for page in pages:
        result.extend(page.get("items", []))
    return result


In [7]:
exercise_2_ready = True  # Set True when your implementation is ready.
if exercise_2_ready:
    assert collect_items([{"items": [1, 2]}, {}, {"items": [3]}]) == [1, 2, 3]
    assert collect_items([]) == []
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: supplied checks passed. Add your own edge cases!


## Build it yourself

Extend the local server with a /health route and test both success and 404 responses. Then design a paginated response with a next-page field. Keep credentials out of notebook cells and saved outputs if you later try an authenticated API.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

Never assume every response is successful JSON. This local HTTP demo is for practice; external services should use HTTPS and their documented authentication and usage rules.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Back to learning notebook](../notebooks/17_http_apis_and_json.ipynb)
